# Camel detector v3: training on Kaggle

1. Add the `camel_v3` dataset with **Add Data**.
2. Turn on the GPU: **Settings → Accelerator → GPU T4**.
3. Run the cells in order: dataset → night/day test → train → evaluate → predictions → video.
4. The model is saved as `best_v3.pt` inside `camel_v3_results.zip` (step 7).
5. To run in the background: **Save Version → Save & Run All (Commit)**. Results appear in the **Output** tab.

In [1]:
!pip -q install -U ultralytics
import ultralytics; ultralytics.checks()

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 5643.8/8156.9 GB disk)


## 1. Find the dataset and write a data file with absolute paths

In [2]:
import glob, os, shutil, csv
WORK = '/kaggle/working'
yamls = [p for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True)]
assert yamls, 'camel_v3 dataset not found: add it with Add Data'
DATA_DIR = os.path.dirname(yamls[0])
print('dataset:', DATA_DIR)
for s in ['train', 'valid', 'test']:
    print(s, len(os.listdir(f'{DATA_DIR}/{s}/images')))
# expected: train 959, valid 188, test 48

DATA_YAML = f'{WORK}/data_v3.yaml'
open(DATA_YAML, 'w').write(f"""path: {DATA_DIR}
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['camel']
""")

dataset: /kaggle/input/datasets/shadenaa/camel-v3/camel_v3
train 959
valid 188
test 48


135

## 2. Night and day test subsets (from `split_manifest.csv`)

In [3]:
cond = {r['file']: r['condition'] for r in csv.DictReader(open(f'{DATA_DIR}/split_manifest.csv')) if r['split'] == 'test'}

def subset(name, keep):
    root = f'{WORK}/{name}'
    shutil.rmtree(root, ignore_errors=True); os.makedirs(f'{root}/images'); os.makedirs(f'{root}/labels')
    n = 0
    for img in glob.glob(f'{DATA_DIR}/test/images/*'):
        b = os.path.splitext(os.path.basename(img))[0]
        if keep(cond.get(b)):
            os.symlink(img, f'{root}/images/{os.path.basename(img)}')
            os.symlink(f'{DATA_DIR}/test/labels/{b}.txt', f'{root}/labels/{b}.txt'); n += 1
    y = f'{root}/data.yaml'
    open(y, 'w').write(f"path: {root}\ntrain: images\nval: images\ntest: images\nnc: 1\nnames: ['camel']\n")
    print(name, n, 'images'); return y

NIGHT_YAML = subset('test_night', lambda c: c == 'night')
DAY_YAML   = subset('test_day',   lambda c: c == 'day')
# expected: 24 and 24

test_night 24 images
test_day 24 images


## 3. Train
Augmentation happens on the fly in YOLO (different every epoch): brightness ±50% for day/night, flips, scale, mosaic, plus light blur from Albumentations.
`imgsz=960` helps far, small camels. About 2–3 hours on a T4.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(
    data=DATA_YAML,
    epochs=100, patience=20,
    imgsz=960, batch=12, device=0, cache=True,
    hsv_v=0.5, hsv_s=0.6, fliplr=0.5, scale=0.5,
    mosaic=1.0, close_mosaic=15,
    project=f'{WORK}/runs', name='camel_v3', exist_ok=True,
)

## 4. Evaluate on the balanced test set (night / day / all)
If `best_v2.pt` was added as data, v2 is evaluated on the same new test for a fair comparison.

In [ ]:
BEST = f'{WORK}/runs/camel_v3/weights/best.pt'
models = {'v3': BEST}
v2 = glob.glob('/kaggle/input/**/best_v2.pt', recursive=True)
if v2: models['v2'] = v2[0]

print(f"{'model':<6}{'imgsz':>6} {'set':<7}{'P':>7}{'R':>7}{'mAP50':>8}{'mAP50-95':>10}")
for mname, w in models.items():
    m = YOLO(w)
    for imgsz in (960, 1280):
        for sname, y in (('all', DATA_YAML), ('night', NIGHT_YAML), ('day', DAY_YAML)):
            r = m.val(data=y, split='test', imgsz=imgsz, conf=0.001, plots=False, verbose=False)
            print(f'{mname:<6}{imgsz:>6} {sname:<7}{r.box.mp:>7.3f}{r.box.mr:>7.3f}{r.box.map50:>8.3f}{r.box.map:>10.3f}')

## 5. Look at the predictions on the test images

In [ ]:
from IPython.display import Image, display
best = YOLO(BEST)
best.predict(source=f'{DATA_DIR}/test/images', imgsz=960, conf=0.25, line_width=2, save=True,
             project=f'{WORK}/runs', name='test_preds', exist_ok=True, verbose=False)
for f in sorted(glob.glob(f'{WORK}/runs/test_preds/*'))[:16]:
    display(Image(f, width=420))

## 6. Optional: video with the horizon boost
Runs only if a video was added as data. Full frame at 960 plus a zoomed horizon strip for small, far camels.

In [ ]:
import cv2, torch, torchvision
videos = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if p.lower().endswith(('.mp4', '.mov', '.avi'))]
print(videos)

def detect(frame, band=(0.30, 0.62), max_h=0.12, conf=0.25):
    H, W = frame.shape[:2]; y0, y1 = int(band[0] * H), int(band[1] * H)
    B, S = [], []
    for b in best.predict(frame, conf=conf, imgsz=960, verbose=False)[0].boxes:
        B.append(b.xyxy[0].tolist()); S.append(float(b.conf))
    for b in best.predict(frame[y0:y1], conf=conf, imgsz=1280, verbose=False)[0].boxes:
        x1, a, x2, c = b.xyxy[0].tolist()
        if (c - a) / H < max_h: B.append([x1, a + y0, x2, c + y0]); S.append(float(b.conf))
    if not B: return []
    keep = torchvision.ops.nms(torch.tensor(B), torch.tensor(S), 0.4).tolist()
    return [(*map(int, B[i]), S[i]) for i in keep]

for v in videos:
    cap = cv2.VideoCapture(v); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w, h = int(cap.get(3)), int(cap.get(4))
    out_path = f'{WORK}/horizon_' + os.path.splitext(os.path.basename(v))[0] + '.mp4'
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    first = None; i = 0
    while True:
        ok, f = cap.read()
        if not ok: break
        bx = detect(f)
        if bx and first is None: first = i / fps
        for x1, y1, x2, y2, s in bx:
            cv2.rectangle(f, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(f, f'{s:.2f}', (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 0), 1)
        out.write(f); i += 1
    cap.release(); out.release()
    print(os.path.basename(v), '| first camel at second:', first, '| saved:', out_path)

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def gt_boxes(img_path, W, H):
    b = os.path.splitext(os.path.basename(img_path))[0]
    out = []
    for l in open(f'{DATA_DIR}/test/labels/{b}.txt').read().splitlines():
        if l.strip():
            _, cx, cy, w, h = map(float, l.split())
            out.append([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H])
    return out

def plain(frame):
    return [(*b.xyxy[0].tolist(), float(b.conf)) for b in best.predict(frame, conf=0.25, imgsz=960, verbose=False)[0].boxes]

stats = {m: {s: [0, 0, 0] for s in ('night', 'day')} for m in ('plain', 'horizon')}   # TP, FP, FN
for img in glob.glob(f'{DATA_DIR}/test/images/*'):
    f = cv2.imread(img); H, W = f.shape[:2]; G = gt_boxes(img, W, H)
    c = cond[os.path.splitext(os.path.basename(img))[0]]
    for m, fn in (('plain', plain), ('horizon', detect)):
        P = sorted(fn(f), key=lambda p: -p[4]); used = set(); tp = 0
        for p in P:
            j = max(range(len(G)), key=lambda k: iou(p, G[k]), default=None)
            if j is not None and j not in used and iou(p, G[j]) >= 0.5: used.add(j); tp += 1
        s = stats[m][c]; s[0] += tp; s[1] += len(P) - tp; s[2] += len(G) - tp

for m in stats:
    for s, (tp, fp, fn) in stats[m].items():
        print(f'{m:<8} {s:<6} P={tp/(tp+fp+1e-9):.3f}  R={tp/(tp+fn+1e-9):.3f}  (TP={tp} FP={fp} FN={fn})')

## 7. Pack the results for download (Output tab)

In [ ]:
res = f'{WORK}/camel_v3_results'
os.makedirs(res, exist_ok=True)
shutil.copy(BEST, f'{res}/best_v3.pt')
for f in ['results.csv', 'results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'PR_curve.png', 'args.yaml']:
    p = f'{WORK}/runs/camel_v3/{f}'
    if os.path.exists(p): shutil.copy(p, res)
shutil.copytree(f'{WORK}/runs/test_preds', f'{res}/test_preds', dirs_exist_ok=True)
shutil.make_archive(f'{WORK}/camel_v3_results', 'zip', res)
print(os.listdir(res))

In [2]:
!pip -q install -U ultralytics
import glob, os, subprocess, cv2, torch, torchvision
from ultralytics import YOLO

W = glob.glob('/kaggle/input/**/best_v3.pt', recursive=True)
assert W, 'best_v3.pt not found: add the notebook output with Add Input'
best = YOLO(W[0])

def detect(frame, band=(0.30, 0.62), max_h=0.12, conf=0.25):
    H, Wd = frame.shape[:2]; y0, y1 = int(band[0] * H), int(band[1] * H)
    B, S = [], []
    for b in best.predict(frame, conf=conf, imgsz=960, verbose=False)[0].boxes:
        B.append(b.xyxy[0].tolist()); S.append(float(b.conf))
    for b in best.predict(frame[y0:y1], conf=conf, imgsz=1280, verbose=False)[0].boxes:
        x1, a, x2, c = b.xyxy[0].tolist()
        if (c - a) / H < max_h: B.append([x1, a + y0, x2, c + y0]); S.append(float(b.conf))
    if not B: return []
    keep = torchvision.ops.nms(torch.tensor(B), torch.tensor(S), 0.4).tolist()
    return [(*map(int, B[i]), S[i]) for i in keep]

videos = [p for p in glob.glob('/kaggle/input/**/*', recursive=True)
          if p.lower().endswith(('.mp4', '.mov', '.avi', '.mkv', '.webm'))]
os.makedirs('/kaggle/working/new_videos', exist_ok=True)
for v in videos:
    name = os.path.splitext(os.path.basename(v))[0]
    tmp, out_path = f'/kaggle/working/new_videos/{name}_raw.mp4', f'/kaggle/working/new_videos/{name}_camels.mp4'
    cap = cv2.VideoCapture(v); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    out = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), fps, (int(cap.get(3)), int(cap.get(4))))
    first = None; i = 0
    while True:
        ok, f = cap.read()
        if not ok: break
        bx = detect(f)
        if bx and first is None: first = i / fps
        for x1, y1, x2, y2, s in bx:
            cv2.rectangle(f, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(f, f'{s:.2f}', (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 0), 1)
        out.write(f); i += 1
    cap.release(); out.release()
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-c:v', 'libx264',
                    '-preset', 'veryfast', '-pix_fmt', 'yuv420p', out_path]); os.remove(tmp)
    print(f'{name} | first camel at second: {first} | saved: {out_path}')

v14044g50000d9dguqfog65qdq8ud2og (1) | first camel at second: 1.0 | saved: /kaggle/working/new_videos/v14044g50000d9dguqfog65qdq8ud2og (1)_camels.mp4
v1c044g50000d9jngn7og65nm5gvo440 | first camel at second: 0.0 | saved: /kaggle/working/new_videos/v1c044g50000d9jngn7og65nm5gvo440_camels.mp4
horizon_42b4ce79-53ee-44ab-a366-e5f7e0617b62 | first camel at second: 5.364720954439907 | saved: /kaggle/working/new_videos/horizon_42b4ce79-53ee-44ab-a366-e5f7e0617b62_camels.mp4
horizon_v14044g50000d9dguqfog65qdq8ud2og | first camel at second: 0.2 | saved: /kaggle/working/new_videos/horizon_v14044g50000d9dguqfog65qdq8ud2og_camels.mp4
horizon_v14044g50000dac69m7og65k3o6ui8s0 | first camel at second: 2.9 | saved: /kaggle/working/new_videos/horizon_v14044g50000dac69m7og65k3o6ui8s0_camels.mp4
